# 📊 Análise de Métricas do Melhor Modelo LSTM

Este notebook analisa as métricas de performance do melhor modelo LSTM treinado para previsão de preços das ações NVIDIA.

## Conteúdo
1. Carregamento de bibliotecas e dados
2. Configuração do modelo
3. Histórico de treinamento (Loss curves)
4. Métricas de regressão (RMSE, MAE, MAPE, R²)
5. Métricas financeiras (Sharpe Ratio, Directional Accuracy)
6. Relatório resumido de performance

In [1]:
# Importar bibliotecas necessárias
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import torch
import json
from pathlib import Path
from datetime import datetime

# Configurações de visualização
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print("✅ Bibliotecas carregadas com sucesso!")

✅ Bibliotecas carregadas com sucesso!


## 1. Carregamento do Modelo e Configuração

Vamos carregar o checkpoint do melhor modelo treinado e extrair suas configurações.

In [ ]:
# Carregar checkpoint do melhor modelo
checkpoint_path = Path("../data/models/checkpoints/best_model.pt")
checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)

# Extrair configuração do modelo
model_config = checkpoint.get('model_config', {})
training_info = {
    'Best Epoch': checkpoint.get('epoch', 'N/A'),
    'Best Val Loss': checkpoint.get('loss', 'N/A'),
    'Timestamp': checkpoint.get('timestamp', 'N/A')
}

# Exibir configuração em formato tabular
config_df = pd.DataFrame([
    {'Parâmetro': 'Input Size', 'Valor': model_config.get('input_size', 1)},
    {'Parâmetro': 'Hidden Size', 'Valor': model_config.get('hidden_size', 128)},
    {'Parâmetro': 'Num Layers', 'Valor': model_config.get('num_layers', 2)},
    {'Parâmetro': 'Dropout', 'Valor': f"{model_config.get('dropout', 0.2):.1%}"},
    {'Parâmetro': 'Bidirectional', 'Valor': model_config.get('bidirectional', False)},
    {'Parâmetro': 'Num Parameters', 'Valor': f"{model_config.get('num_parameters', 0):,}"},
])

print("=" * 60)
print("🧠 CONFIGURAÇÃO DO MODELO LSTM")
print("=" * 60)
print(config_df.to_string(index=False))
print()
print("=" * 60)
print("📈 INFORMAÇÕES DO TREINAMENTO")
print("=" * 60)
for key, value in training_info.items():
    if isinstance(value, float):
        print(f"{key}: {value:.6f}")
    else:
        print(f"{key}: {value}")

🧠 CONFIGURAÇÃO DO MODELO LSTM
     Parâmetro   Valor
    Input Size       1
   Hidden Size     128
    Num Layers       2
       Dropout   20.0%
 Bidirectional   False
Num Parameters 199,297

📈 INFORMAÇÕES DO TREINAMENTO
Best Epoch: 47
Best Val Loss: 0.000605
Timestamp: 2026-01-31T17:08:22.945848


## 2. Carregamento das Métricas do MLflow

Vamos carregar o histórico completo de métricas do treinamento armazenado no MLflow.

In [ ]:
# Conectar ao banco MLflow
mlflow_db_path = Path("../data/mlruns/mlflow.db")
conn = sqlite3.connect(mlflow_db_path)

# Carregar todas as métricas do último run
metrics_df = pd.read_sql('''
    SELECT m.key, m.value, m.step, m.timestamp, r.run_uuid
    FROM metrics m
    JOIN runs r ON m.run_uuid = r.run_uuid
    ORDER BY m.step, m.key
''', conn)

# Identificar o run mais recente
latest_run = metrics_df['run_uuid'].iloc[-1] if len(metrics_df) > 0 else None
print(f"📌 Run ID: {latest_run}")

📌 Run ID: 5747312b7a014d5b9cf58802018fe5b0

📊 Total de métricas: 1168
  - Training: 550 registros
  - Validation: 550 registros
  - Test: 16 registros


## 3. Métricas Finais de Teste

Resumo das métricas de performance no conjunto de teste.

In [4]:
# Extrair métricas de teste finais
test_results = {}
for _, row in test_metrics.iterrows():
    metric_name = row['key'].replace('test_', '')
    test_results[metric_name] = row['value']

# Criar DataFrame com métricas organizadas
metrics_categories = {
    'Métricas de Erro': {
        'Loss (MSE)': test_results.get('loss', None),
        'RMSE': test_results.get('rmse', None),
        'MAE': test_results.get('mae', None),
        'Max Error': test_results.get('max_error', None),
    },
    'Métricas Percentuais': {
        'MAPE (%)': test_results.get('mape', None),
        'sMAPE (%)': test_results.get('smape', None),
    },
    'Métricas de Ajuste': {
        'R² Score': test_results.get('r2', None),
        'Correlação': test_results.get('correlation', None),
        'MASE': test_results.get('mase', None),
        "Theil's U": test_results.get('theils_u', None),
    },
    'Métricas Direcionais': {
        'Directional Accuracy (%)': test_results.get('directional_accuracy', None),
        'Win Rate (%)': test_results.get('win_rate', None),
        'Win/Loss Ratio': test_results.get('win_loss_ratio', None),
    },
    'Métricas Financeiras': {
        'Sharpe Ratio': test_results.get('sharpe_ratio', None),
        'Profit Factor': test_results.get('profit_factor', None),
        'Max Drawdown (%)': test_results.get('max_drawdown', None),
    }
}

# Exibir métricas por categoria
print("=" * 70)
print("📊 MÉTRICAS DE PERFORMANCE NO CONJUNTO DE TESTE")
print("=" * 70)

for category, metrics in metrics_categories.items():
    print(f"\n📌 {category}")
    print("-" * 40)
    for name, value in metrics.items():
        if value is not None:
            if abs(value) < 0.01:
                print(f"  {name}: {value:.6f}")
            elif abs(value) > 100:
                print(f"  {name}: {value:.2f}")
            else:
                print(f"  {name}: {value:.4f}")

📊 MÉTRICAS DE PERFORMANCE NO CONJUNTO DE TESTE

📌 Métricas de Erro
----------------------------------------
  Loss (MSE): 0.001741
  RMSE: 0.0417
  MAE: 0.0355
  Max Error: 0.1353

📌 Métricas Percentuais
----------------------------------------
  MAPE (%): 4.9450
  sMAPE (%): 5.0612

📌 Métricas de Ajuste
----------------------------------------
  R² Score: 0.9064
  Correlação: 0.9728
  MASE: 2.4174
  Theil's U: 2.1137

📌 Métricas Direcionais
----------------------------------------
  Directional Accuracy (%): 46.3855
  Win Rate (%): 46.3855
  Win/Loss Ratio: 0.9794

📌 Métricas Financeiras
----------------------------------------
  Sharpe Ratio: -0.9321
  Profit Factor: 0.8473
  Max Drawdown (%): 95.9029


## 4. Curvas de Treinamento (Loss)

Visualização da evolução do loss durante o treinamento.

In [5]:
# Preparar dados de histórico de treinamento
def pivot_metrics_by_step(metrics_df, prefix):
    """Transforma métricas em DataFrame pivot por step (epoch)."""
    filtered = metrics_df[metrics_df['key'].str.startswith(prefix)].copy()
    filtered['metric_name'] = filtered['key'].str.replace(prefix, '')
    pivot = filtered.pivot_table(index='step', columns='metric_name', values='value', aggfunc='first')
    return pivot

train_history = pivot_metrics_by_step(run_metrics, 'train_')
val_history = pivot_metrics_by_step(run_metrics, 'val_')

# Plotar curvas de Loss
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Loss (MSE)', 'RMSE', 'MAE', 'R²'),
    vertical_spacing=0.12,
    horizontal_spacing=0.1
)

# Loss curve
fig.add_trace(
    go.Scatter(x=train_history.index, y=train_history['loss'], name='Train Loss', 
               line=dict(color='#2E86AB', width=2)),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=val_history.index, y=val_history['loss'], name='Val Loss',
               line=dict(color='#E94F37', width=2)),
    row=1, col=1
)

# RMSE curve
fig.add_trace(
    go.Scatter(x=train_history.index, y=train_history['rmse'], name='Train RMSE',
               line=dict(color='#2E86AB', width=2), showlegend=False),
    row=1, col=2
)
fig.add_trace(
    go.Scatter(x=val_history.index, y=val_history['rmse'], name='Val RMSE',
               line=dict(color='#E94F37', width=2), showlegend=False),
    row=1, col=2
)

# MAE curve
fig.add_trace(
    go.Scatter(x=train_history.index, y=train_history['mae'], name='Train MAE',
               line=dict(color='#2E86AB', width=2), showlegend=False),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(x=val_history.index, y=val_history['mae'], name='Val MAE',
               line=dict(color='#E94F37', width=2), showlegend=False),
    row=2, col=1
)

# R² curve
fig.add_trace(
    go.Scatter(x=train_history.index, y=train_history['r2'], name='Train R²',
               line=dict(color='#2E86AB', width=2), showlegend=False),
    row=2, col=2
)
fig.add_trace(
    go.Scatter(x=val_history.index, y=val_history['r2'], name='Val R²',
               line=dict(color='#E94F37', width=2), showlegend=False),
    row=2, col=2
)

fig.update_layout(
    title='📈 Histórico de Treinamento - Métricas por Época',
    height=600,
    template='plotly_white',
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig.update_xaxes(title_text="Época", row=2, col=1)
fig.update_xaxes(title_text="Época", row=2, col=2)

fig.show()

## 5. Comparação Train vs Validation (Overfitting Analysis)

Análise de overfitting comparando métricas de treino e validação.

In [6]:
# Calcular gap entre train e val (indicador de overfitting)
last_epoch = train_history.index.max()
final_train = train_history.loc[last_epoch]
final_val = val_history.loc[last_epoch]

# Criar comparação
comparison_data = []
for metric in ['loss', 'rmse', 'mae', 'r2', 'mape']:
    if metric in final_train and metric in final_val:
        train_val = final_train[metric]
        val_val = final_val[metric]
        gap = val_val - train_val
        gap_pct = (gap / train_val * 100) if train_val != 0 else 0
        comparison_data.append({
            'Métrica': metric.upper(),
            'Train': train_val,
            'Validation': val_val,
            'Gap': gap,
            'Gap (%)': gap_pct
        })

comparison_df = pd.DataFrame(comparison_data)

# Plotar comparação
fig = go.Figure()

metrics_names = comparison_df['Métrica'].tolist()
x = np.arange(len(metrics_names))

fig.add_trace(go.Bar(
    name='Train',
    x=metrics_names,
    y=comparison_df['Train'],
    marker_color='#2E86AB',
    text=comparison_df['Train'].apply(lambda x: f'{x:.4f}'),
    textposition='auto'
))

fig.add_trace(go.Bar(
    name='Validation',
    x=metrics_names,
    y=comparison_df['Validation'],
    marker_color='#E94F37',
    text=comparison_df['Validation'].apply(lambda x: f'{x:.4f}'),
    textposition='auto'
))

fig.update_layout(
    title='📊 Comparação Final: Train vs Validation',
    barmode='group',
    template='plotly_white',
    height=400,
    xaxis_title='Métrica',
    yaxis_title='Valor'
)

fig.show()

# Tabela de comparação
print("\n📋 Tabela de Comparação (Última Época):")
print(comparison_df.to_string(index=False))


📋 Tabela de Comparação (Última Época):
Métrica   Train  Validation      Gap   Gap (%)
   LOSS  0.0000      0.0008   0.0008 2643.6545
   RMSE  0.0053      0.0279   0.0226  423.7991
    MAE  0.0035      0.0206   0.0171  484.3874
     R2  0.9813      0.9669  -0.0144   -1.4637
   MAPE 19.9136      5.5047 -14.4090  -72.3573


## 6. Métricas de Teste - Visualização

Gráfico de barras com todas as métricas do conjunto de teste.

In [7]:
# Visualização das métricas de teste
fig = make_subplots(
    rows=2, cols=2,
    specs=[[{'type': 'indicator'}, {'type': 'indicator'}],
           [{'type': 'bar', 'colspan': 2}, None]],
    subplot_titles=('', '', 'Métricas de Erro'),
    vertical_spacing=0.15
)

# Indicador R²
fig.add_trace(
    go.Indicator(
        mode="gauge+number",
        value=test_results.get('r2', 0) * 100,
        title={'text': "R² Score (%)"},
        gauge={
            'axis': {'range': [0, 100]},
            'bar': {'color': "#2E86AB"},
            'steps': [
                {'range': [0, 50], 'color': "#ffcccc"},
                {'range': [50, 80], 'color': "#ffffcc"},
                {'range': [80, 100], 'color': "#ccffcc"}
            ],
            'threshold': {
                'line': {'color': "red", 'width': 4},
                'thickness': 0.75,
                'value': 90
            }
        }
    ),
    row=1, col=1
)

# Indicador Correlação
fig.add_trace(
    go.Indicator(
        mode="gauge+number",
        value=test_results.get('correlation', 0) * 100,
        title={'text': "Correlação (%)"},
        gauge={
            'axis': {'range': [0, 100]},
            'bar': {'color': "#A23B72"},
            'steps': [
                {'range': [0, 50], 'color': "#ffcccc"},
                {'range': [50, 80], 'color': "#ffffcc"},
                {'range': [80, 100], 'color': "#ccffcc"}
            ]
        }
    ),
    row=1, col=2
)

# Bar chart de métricas de erro
error_metrics = {
    'RMSE': test_results.get('rmse', 0),
    'MAE': test_results.get('mae', 0),
    'MAPE (%)': test_results.get('mape', 0),
    'sMAPE (%)': test_results.get('smape', 0),
}

fig.add_trace(
    go.Bar(
        x=list(error_metrics.keys()),
        y=list(error_metrics.values()),
        marker_color=['#2E86AB', '#A23B72', '#F18F01', '#C73E1D'],
        text=[f'{v:.4f}' for v in error_metrics.values()],
        textposition='auto'
    ),
    row=2, col=1
)

fig.update_layout(
    title='📊 Dashboard de Métricas de Teste',
    height=600,
    template='plotly_white',
    showlegend=False
)

fig.show()

## 7. Métricas Direcionais e Financeiras

Análise das métricas específicas para trading e finanças.

In [9]:
# Métricas direcionais e financeiras
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'domain'}, {'type': 'xy'}]],
    subplot_titles=('Directional Accuracy', 'Métricas Financeiras')
)

# Pie chart para directional accuracy
dir_acc = test_results.get('directional_accuracy', 50)
fig.add_trace(
    go.Pie(
        labels=['Direção Correta', 'Direção Incorreta'],
        values=[dir_acc, 100 - dir_acc],
        marker_colors=['#2E86AB', '#E94F37'],
        hole=0.4,
        textinfo='percent+label'
    ),
    row=1, col=1
)

# Bar chart de métricas financeiras
financial_metrics = {
    'Sharpe Ratio': test_results.get('sharpe_ratio', 0),
    'Profit Factor': test_results.get('profit_factor', 0),
    'Win/Loss Ratio': test_results.get('win_loss_ratio', 0),
}

colors = ['#2E86AB' if v >= 0 else '#E94F37' for v in financial_metrics.values()]

fig.add_trace(
    go.Bar(
        x=list(financial_metrics.keys()),
        y=list(financial_metrics.values()),
        marker_color=colors,
        text=[f'{v:.3f}' for v in financial_metrics.values()],
        textposition='auto'
    ),
    row=1, col=2
)

# Adicionar linha de referência em 1.0 para métricas de ratio (usando add_shape)
fig.add_shape(
    type="line",
    x0=-0.5, x1=2.5, y0=1.0, y1=1.0,
    line=dict(color="gray", width=2, dash="dash"),
    xref='x2', yref='y2'
)
fig.add_annotation(
    x=2.5, y=1.0, text="Break-even",
    showarrow=False, font=dict(color="gray"),
    xref='x2', yref='y2'
)

fig.update_layout(
    title='📈 Métricas Direcionais e Financeiras',
    height=400,
    template='plotly_white'
)

fig.show()

# Análise de Max Drawdown
max_dd = test_results.get('max_drawdown', 0)
print(f"\n⚠️ Max Drawdown: {max_dd:.2f}%")
if max_dd > 50:
    print("   ⚠️ Alto risco - Drawdown significativo durante o período de teste")


⚠️ Max Drawdown: 95.90%
   ⚠️ Alto risco - Drawdown significativo durante o período de teste


## 8. Evolução das Métricas por Época (Heatmap)

In [10]:
# Heatmap de evolução das métricas de validação
metrics_to_show = ['loss', 'rmse', 'mae', 'r2', 'correlation']
val_subset = val_history[metrics_to_show].copy()

# Normalizar para visualização (0-1)
val_normalized = (val_subset - val_subset.min()) / (val_subset.max() - val_subset.min())

fig = go.Figure(data=go.Heatmap(
    z=val_normalized.T.values,
    x=val_normalized.index,
    y=metrics_to_show,
    colorscale='RdYlGn_r',
    text=val_subset.T.values.round(4),
    texttemplate='%{text}',
    textfont={"size": 8},
    hoverongaps=False
))

fig.update_layout(
    title='🗺️ Heatmap: Evolução das Métricas de Validação por Época (Normalizado)',
    xaxis_title='Época',
    yaxis_title='Métrica',
    height=350,
    template='plotly_white'
)

fig.show()

## 9. 📝 Relatório Resumido de Performance

Análise consolidada do modelo com interpretação dos resultados.

In [ ]:
# Salvar métricas em JSON para referência futura
output_report = {
    'model_config': model_config,
    'training_info': {
        'best_epoch': training_info['Best Epoch'],
        'best_val_loss': float(training_info['Best Val Loss']),
        'timestamp': training_info['Timestamp']
    },
    'test_metrics': {k: float(v) for k, v in test_results.items()},
    'run_id': latest_run
}

report_path = Path('../data/outputs/model_performance_report.json')
report_path.parent.mkdir(parents=True, exist_ok=True)

with open(report_path, 'w') as f:
    json.dump(output_report, f, indent=2)

print(f"\n✅ Relatório salvo em: {report_path.absolute()}")

📊 RELATÓRIO DE PERFORMANCE DO MODELO LSTM - NVIDIA STOCK PREDICTION

🗓️  Data do Treinamento: 2026-01-31T17:08:22.945848
📌 Run ID: 5747312b7a014d5b9cf58802018fe5b0
🔄 Épocas Treinadas: 47

             Métrica  Valor         Interpretação                                           Comentário
            R² Score 0.9064           ✅ Excelente        O modelo explica 90.6% da variância dos dados
          Correlação 0.9728           ✅ Excelente Forte relação linear entre previsões e valores reais
                MAPE  4.94%           ✅ Excelente                        Erro médio percentual de 4.9%
  RMSE (normalizado) 0.0417          ⚠️ Aceitável          Erro quadrático médio em escala normalizada
Directional Accuracy  46.4% ❌ Necessita melhorias         Acurácia na previsão da direção do movimento
        Sharpe Ratio -0.932 ❌ Necessita melhorias Retorno ajustado ao risco (>1 é bom, >2 é excelente)

📋 RESUMO EXECUTIVO

🧠 ARQUITETURA DO MODELO:
   - Tipo: LSTM (Long Short-Term Memory)
   -

In [13]:
# Fechar conexão com o banco
conn.close()

# Salvar métricas em JSON para referência futura
output_report = {
    'model_config': model_config,
    'training_info': {
        'best_epoch': training_info['Best Epoch'],
        'best_val_loss': float(training_info['Best Val Loss']),
        'timestamp': training_info['Timestamp']
    },
    'test_metrics': {k: float(v) for k, v in test_results.items()},
    'run_id': latest_run
}

report_path = Path('../outputs/model_performance_report.json')
report_path.parent.mkdir(parents=True, exist_ok=True)

with open(report_path, 'w') as f:
    json.dump(output_report, f, indent=2)

print(f"\n✅ Relatório salvo em: {report_path.absolute()}")


✅ Relatório salvo em: /home/lucas/nvidia-lstm-forecast/notebooks/../outputs/model_performance_report.json
